<center>
<img src="https://supportvectors.ai/logo-poster-transparent.png" width="400px" style="opacity:0.7">
</center>

## Working with Qdrant

In [1]:
%run supportvectors-common.ipynb


<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors. All rights reserved. <blockquote>This notebook is the intellectual property of SupportVectors, and part of its training material. 
Only the participants in SupportVectors workshops are allowed to study the notebooks for educational purposes currently, but is prohibited from copying or using it for any other purposes without written permission.

<b> These notebooks are chapters and sections from Asif Qamar's textbook that he is writing on Data Science. So we request you to not circulate the material to others.</b>
 </blockquote>
 <hr/>
</div>



First make sure your qdrant docker is running using the below steps:

- **If you are on a Mac or Windows, start your Docker Desktop first.**
- Then follow the below steps
- On a Mac, you could also create a volume through the Docker Desktop GUI and then associate the named volume in the second step below

```bash
# Pull Qdrant image
docker pull qdrant/qdrant

# Run Qdrant (on linux)
docker run -p 6333:6333 -p 6334:6334 \
    -v $(pwd)/qdrant_storage:/qdrant/storage \
    qdrant/qdrant

## if on Mac (replace <my_volume> with the volume you created in the GUI)
docker run -p 6333:6333 -p 6334:6334 \
    -v <my_volume>:/qdrant/storage \
    qdrant/qdrant
```


### Imports

In [2]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from qdrant_client.models import Filter, FieldCondition, MatchValue
from rich import print as rprint
import numpy as np
import requests

### Connect to Qdrant and create a collection

In [4]:

# Connect to Qdrant
client = QdrantClient("localhost", port=6333)

# Create collection
collection_name = "my_collection"
vector_size = 128

# Check if collection exists
try:
    client.get_collection(collection_name)
    print(f"Collection '{collection_name}' already exists. Skipping creation.")
except Exception:
    print(f"Collection '{collection_name}' does not exist. Creating now.")
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=vector_size,
            distance=Distance.COSINE
        )
    )

Collection 'my_collection' already exists. Skipping creation.


### Basic operations

#### Insert vectors

In [5]:
# Prepare data
vector1 = np.random.rand(128).astype(np.float32)
vector2 = np.random.rand(128).astype(np.float32)
points = [
    PointStruct(
        id=1,
        vector=vector1,
        payload={"text": "sample text", "category": "news"}
    ),
    PointStruct(
        id=2,
        vector=vector2,
        payload={"text": "another text", "category": "sports"}
    )
]

# Insert points
client.upsert(
    collection_name=collection_name,
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

#### Search vectors

In [6]:
# Basic search
search_result = client.query_points(
    collection_name=collection_name,
    query=vector1,
    limit=10
).points


In [7]:
rprint(search_result)

[
    ScoredPoint(
        id=1,
        version=0,
        score=0.9999999,
        payload={'text': 'sample text', 'category': 'news'},
        vector=None,
        shard_key=None,
        order_value=None
    ),
    ScoredPoint(
        id=2,
        version=0,
        score=0.73362184,
        payload={'text': 'another text', 'category': 'sports'},
        vector=None,
        shard_key=None,
        order_value=None
    )
]

In [8]:

# Search with filtering

search_result = client.query_points(
    collection_name=collection_name,
    query=vector1,
    query_filter=Filter(
        must=[
            FieldCondition(
                key="category",
                match=MatchValue(value="sports")
            )
        ]
    ),
    limit=10
).points

In [9]:
rprint(search_result)

[
    ScoredPoint(
        id=2,
        version=0,
        score=0.73362184,
        payload={'text': 'another text', 'category': 'sports'},
        vector=None,
        shard_key=None,
        order_value=None
    )
]

#### **Get Input Book Text from Gutenberg, and format it**

In [10]:
# Get the text from gutenberg for "The Great Gatsby"
book_text_full = requests.get("https://www.gutenberg.org/cache/epub/64317/pg64317.txt").text
print(book_text_full[505:600])  # Print few characters



Title: The Great Gatsby

Author: F. Scott Fitzgerald

Release date: January 17, 2021 [eB


In [11]:
# Stripping the Gutenberg Text etc. below, 

# Removes the first part
book_text = book_text_full[900:]  
print(f"\nBook Start Part: {book_text[0:100]}")

total_length = len(book_text)
print(f"\nTotal book length: {total_length}")
print(f"\nBook End Part: {book_text[277120:277190]}")

# Removes the end part. Now book conains the related text(content) only.
book_text = book_text[0:277120]  


Book Start Part: 
                           The Great Gatsby
                                  by
                

Total book length: 295958

Book End Part: 

*** END OF THE PROJECT GUTENBERG EBOOK THE GREAT GATSBY ***


 


#### **Initialize the `semchunk` chunker for chunking our text book**

In [12]:
import semchunk
chunk_size=512
chunker = semchunk.chunkerify('gpt-4', chunk_size)

# Chunk the text
chunks = chunker(book_text)

total_chunks = len(chunks)
print(f"Total Number of Chunks: {total_chunks}")

print(f"Example Chunk-1:{chunks[1]}")

Total Number of Chunks: 148
Example Chunk-1:                                  I

In my younger and more vulnerable years my father gave me some advice
that I’ve been turning over in my mind ever since.

“Whenever you feel like criticizing anyone,” he told me, “just
remember that all the people in this world haven’t had the advantages
that you’ve had.”

He didn’t say any more, but we’ve always been unusually communicative
in a reserved way, and I understood that he meant a great deal more
than that. In consequence, I’m inclined to reserve all judgements, a
habit that has opened up many curious natures to me and also made me
the victim of not a few veteran bores. The abnormal mind is quick to
detect and attach itself to this quality when it appears in a normal
person, and so it came about that in college I was unjustly accused of
being a politician, because I was privy to the secret griefs of wild,
unknown men. Most of the confidences were unsought—frequently I have
feigned sleep, preocc

#### **Load a sentence-transformer model for vector embedding generation**

In [13]:
# Load embedding model
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [14]:
# Generate embeddings
embeddings = [model.encode(chunk).tolist() for chunk in chunks]

#### **Create a collection in qdrant, and upload the embedding-vectors to it**

In [15]:
collection_name = "Book-1"

In [16]:
model.get_sentence_embedding_dimension()

384

In [17]:
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=model.get_sentence_embedding_dimension(),  # Vector size is defined by used model
        distance=Distance.COSINE,
    ),
)

True

In [18]:
# Upload chunks with embeddings
points = [
    PointStruct(id=i, vector=embeddings[i], payload={"text": chunks[i]})
    for i in range(len(chunks))
]

client.upsert(collection_name=collection_name, points=points)

print("Embeddings stored successfully!")

Embeddings stored successfully!


## **Search the Vector DB**

In [ ]:

# Standard mode using ANN
def query_qdrant_standard(query_text, top_k = 2):
    
    results = client.query_points(
        collection_name=collection_name,
        query=model.encode(query_text).tolist(),
        limit=top_k,
    ).points

    return results


#### **QUERY**

In [20]:
query_text = "Find passages where Gatsby expresses his love for Daisy"

In [21]:
results = query_qdrant_standard(query_text)

for result in results:
    print("\nSCORE: ", result.score)
    payload = result.payload['text']
    payload_stripped= payload.replace("\n", " ").replace("\r", "").replace("  ", " ")

    print(payload_stripped)


SCORE:  0.71344906
But he didn’t despise himself and it didn’t turn out as he had imagined. He had intended, probably, to take what he could and go—but now he found that he had committed himself to the following of a grail. He knew that Daisy was extraordinary, but he didn’t realize just how extraordinary a “nice” girl could be. She vanished into her rich house, into her rich, full life, leaving Gatsby—nothing. He felt married to her, that was all. When they met again, two days later, it was Gatsby who was breathless, who was, somehow, betrayed. Her porch was bright with the bought luxury of star-shine; the wicker of the settee squeaked fashionably as she turned toward him and he kissed her curious and lovely mouth. She had caught a cold, and it made her voice huskier and more charming than ever, and Gatsby was overwhelmingly aware of the youth and mystery that wealth imprisons and preserves, of the freshness of many clothes, and of Daisy, gleaming like silver, safe and proud above th